### Purpose

This notebook performs focused descriptive analytics on the approved Gold v1 data products.


The objective is to identify meaningful patterns and relationships that can be explored and presented through Power BI. This notebook does not modify Bronze, Silver, or Gold data and does not create additional Gold tables.

The analysis is descriptive. Observed relationships are not interpreted as causal or predictive unless explicitly supported by additional analysis.

### Analytical areas

The analysis focuses on four areas:

1. User behavior
- How concentrated are transaction activity and spending across users?
- What behavioral patterns are visible?
- How does observed fraud vary across users?

2. Card behavior
- How does activity vary across cards?
- How many cards are inactive?
- How does observed fraud vary at card level?

3. Merchant behavior
- How concentrated are transactions and spending across merchants?
- How does online behavior vary?
- How is observed fraud distributed across merchants?
- What geographic patterns are visible at transaction level?

4. Fraud features
- How does observed fraud relate descriptively to historical user fraud?
- How does observed fraud relate descriptively to historical card fraud?
- How does observed fraud relate descriptively to historical merchant fraud?
- Do historical transaction volumes differ between fraudulent and non-fraudulent transactions?

### Power BI objective

The outputs of this notebook will be used to identify a small number of meaningful measures, relationships, and visualizations for the Power BI analytics layer.

No unnecessary Gold tables or additional persistent data products will be created.

### Gold v1 data sources

The analysis uses the four approved Gold v1 data products:

| Gold object | Grain | Purpose |
|---|---|---|
| gold_user_behavior | user_id | User-level behavior |
| gold_card_behavior | (user_id, card_id) | Card-level behavior |
| gold_merchant_behavior | merchant_id | Merchant-level behavior |
| gold_fraud_features | One row per Silver transaction | Transaction-level fraud context |

The Gold tables are treated as the authoritative analytical inputs for this notebook.

### Load Gold v1

In [0]:
from pyspark.sql import functions as F
# Load approved Gold v1 data products
user_df = spark.table("credit_card_project.gold.gold_user_behavior")
card_df = spark.table("credit_card_project.gold.gold_card_behavior")
merchant_df = spark.table("credit_card_project.gold.gold_merchant_behavior")
fraud_df = spark.table("credit_card_project.gold.gold_fraud_features")


In [0]:
#lightweight structural check
print("User behavior columns:")
print(user_df.columns)

print("\nCard behavior columns:")
print(card_df.columns)

print("\nMerchant behavior columns:")
print(merchant_df.columns)

print("\nFraud features columns:")
print(fraud_df.columns)


### 1. User Behavior

This section examines transaction activity, spending, and observed fraud at user level.

The analysis focuses on:
- distribution of transaction activity across users
- distribution of spending across users
- concentration of transactions and spending
- relationship between activity, spending, and observed fraud

The objective is to identify descriptive patterns that are useful for the Power BI analytics layer.

In [0]:
# user-level summary statistics
user_summary = user_df.select(
    F.count("*").alias("users"),
    F.sum("total_transactions").alias("transactions"),
    F.sum("total_spend").alias("total_spend"),
    F.avg("total_transactions").alias("avg_transactions_per_user"),
    F.expr("percentile_approx(total_transactions, 0.5)").alias("median_transactions_per_user"),
    F.avg("total_spend").alias("avg_spend_per_user"),
    F.expr("percentile_approx(total_spend, 0.5)").alias("median_spend_per_user"),
    F.avg("fraud_rate").alias("avg_user_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.5)").alias("median_user_fraud_rate")
)

display(user_summary)


In [0]:
#distribution of transaction activity
transaction_distribution = user_df.select(
    F.min("total_transactions").alias("min_transactions"),
    F.expr("percentile_approx(total_transactions, 0.25)").alias("p25_transactions"),
    F.expr("percentile_approx(total_transactions, 0.50)").alias("p50_transactions"),
    F.expr("percentile_approx(total_transactions, 0.75)").alias("p75_transactions"),
    F.expr("percentile_approx(total_transactions, 0.90)").alias("p90_transactions"),
    F.expr("percentile_approx(total_transactions, 0.95)").alias("p95_transactions"),
    F.expr("percentile_approx(total_transactions, 0.99)").alias("p99_transactions"),
    F.max("total_transactions").alias("max_transactions")
)

display(transaction_distribution)


In [0]:
#distribution of spending
spend_distribution = user_df.select(
    F.min("total_spend").alias("min_spend"),
    F.expr("percentile_approx(total_spend, 0.25)").alias("p25_spend"),
    F.expr("percentile_approx(total_spend, 0.50)").alias("p50_spend"),
    F.expr("percentile_approx(total_spend, 0.75)").alias("p75_spend"),
    F.expr("percentile_approx(total_spend, 0.90)").alias("p90_spend"),
    F.expr("percentile_approx(total_spend, 0.95)").alias("p95_spend"),
    F.expr("percentile_approx(total_spend, 0.99)").alias("p99_spend"),
    F.max("total_spend").alias("max_spend")
)

display(spend_distribution)


In [0]:
#calculate the cumulative share of transactions and spending contributed by the highest-activity users

# The user-level table contains only 2,000 rows,
# so collecting the small aggregated result is appropriate here.
user_concentration = (
    user_df
    .select(
        "user_id",
        "total_transactions",
        "total_spend"
    )
    .orderBy(F.desc("total_transactions"))
    .collect()
)

total_transactions = sum(row["total_transactions"] for row in user_concentration)
total_spend = sum(row["total_spend"] for row in user_concentration)

user_count = len(user_concentration)
concentration_levels = [1, 5, 10, 20]
rows = []

for pct in concentration_levels:
    n = max(1, int(user_count * pct / 100))
    top_users = user_concentration[:n]
    transaction_share = (
        sum(row["total_transactions"] for row in top_users)
        / total_transactions
    )

    top_spend_users = sorted(
        user_concentration,
        key=lambda row: row["total_spend"],
        reverse=True
    )[:n]

    spend_share = (
        sum(row["total_spend"] for row in top_spend_users)
        / total_spend
    )

    rows.append(
        (
            pct,
            n,
            transaction_share,
            spend_share
        )
    )

concentration_df = spark.createDataFrame(
    rows,
    [
        "top_user_pct",
        "user_count",
        "transaction_share",
        "spend_share"
    ]
)
display(concentration_df)


In [0]:
#user behavioral relationships: transactions vs. spending vs. fraud rate

user_relationships = user_df.select(
    "user_id",
    "total_transactions",
    "total_spend",
    "fraud_count",
    "fraud_rate",
    "unique_cards",
    "unique_merchants",
    "online_transaction_pct",
    "weekend_transaction_pct",
    "night_transaction_pct"
)

display(user_relationships)


In [0]:
#user behavioral relationships
import pandas as pd

correlation_columns = [
    "total_transactions",
    "total_spend",
    "fraud_count",
    "fraud_rate",
    "unique_cards",
    "unique_merchants",
    "online_transaction_pct",
    "weekend_transaction_pct",
    "night_transaction_pct"
]

user_corr_pd = (
    user_relationships
    .select(correlation_columns)
    .toPandas()
    .corr()
)

display(user_corr_pd)

What stands out:

A few relationships are strong enough to investigate further:
- Transactions ↔ spending: 0.851: Users with more transactions generally have higher total spending. This is unsurprising but useful for a Power BI scatter plot.

- Transactions ↔ unique merchants: 0.763: Higher-activity users tend to transact with more distinct merchants.

- Spending ↔ unique merchants: 0.787: This is another fairly strong relationship and potentially more interesting from a behavioral perspective.

- Fraud count ↔ fraud rate: 0.752: This needs some caution. fraud_rate is calculated from fraud count and transaction count, so a relationship between those variables is partly mathematical rather than an independent behavioral discovery.

- Transaction activity ↔ fraud rate: 0.102

- Spending ↔ fraud rate: 0.102: These are relatively weak relationships at user level. That's useful because it means we shouldn't build a story suggesting that high-volume/high-spend users necessarily have higher observed fraud rates.

- The weekend/night percentages have essentially no linear relationship with fraud rate in this user-level analysis.


What I'd carry forward to Power BI

I'd tentatively keep:
- User transaction volume vs. total spending — scatter plot
- User transaction volume vs. number of unique merchants — potentially useful
- User spending concentration — definitely useful
- Fraud rate distribution across users — useful, but probably as a distribution rather than a ranking

In [0]:
 #fraud-rate distribution to determine whether fraud is distributed among users or concentrated in a smaller population
fraud_rate_distribution = user_df.select(
    F.min("fraud_rate").alias("min_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.25)").alias("p25_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.50)").alias("p50_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.75)").alias("p75_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.90)").alias("p90_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.95)").alias("p95_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.99)").alias("p99_fraud_rate"),
    F.max("fraud_rate").alias("max_fraud_rate")
)

display(fraud_rate_distribution)


In [0]:
fraud_users = user_df.select(
    F.sum(
        F.when(F.col("fraud_count") > 0, 1).otherwise(0)
    ).alias("users_with_fraud"),
    F.count("*").alias("total_users")
)

display(fraud_users)


User-level findings so far
- 1,343 of 2,000 users (67.2%) have at least one observed fraudulent transaction.
- The median user fraud rate is ~0.064%.
- The 75th percentile is ~0.163%.
- The 95th percentile is ~0.385%.
- The maximum observed user fraud rate is ~1.77%.
- Most users therefore have relatively low observed fraud rates, despite fraud being present across a substantial portion of the user population.


### 2. Card Behavior

This section examines activity, card type, and observed fraud at card level.

The analysis focuses on:
- active versus inactive cards
- transaction and spending activity by card type
- fraud distribution across card types
- relationship between card activity and observed fraud

The approved gold_card_behavior population is driven by the Bronze card population. Inactive cards are retained, and percentage metrics for inactive cards remain NULL.

In [0]:
card_population = card_df.select(
    F.count("*").alias("cards"),
    F.countDistinct("user_id").alias("users"),
    F.sum(
        F.when(F.col("total_transactions") > 0, 1).otherwise(0)
    ).alias("active_cards"),
    F.sum(
        F.when(F.col("total_transactions") == 0, 1).otherwise(0)
    ).alias("inactive_cards")
)

display(card_population)

In [0]:
display(
    card_df
    .groupBy("card_type")
    .agg(
        F.count("*").alias("cards"),
        F.sum("total_transactions").alias("transactions"),
        F.sum("total_spend").alias("spend"),
        F.sum("fraud_count").alias("fraud_count"),
        F.avg("fraud_rate").alias("avg_card_fraud_rate")
    )
    .orderBy(F.desc("transactions"))
)


In [0]:
#card-type analytical comparison
card_type_analysis = (
    card_df
    .groupBy("card_type")
    .agg(
        F.count("*").alias("cards"),
        F.sum("total_transactions").alias("transactions"),
        F.sum("total_spend").alias("spend"),
        F.sum("fraud_count").alias("fraud_count"),
        F.sum("total_transactions").alias("fraud_rate_denominator"),
        F.avg("fraud_rate").alias("avg_card_fraud_rate"),
        F.avg("total_transactions").alias("avg_transactions_per_card"),
        F.expr("percentile_approx(total_transactions, 0.50)")
            .alias("median_transactions_per_card"),
        F.avg("total_spend").alias("avg_spend_per_card"),
        F.expr("percentile_approx(total_spend, 0.50)")
            .alias("median_spend_per_card")
    )
    .withColumn(
        "transaction_weighted_fraud_rate",
        F.col("fraud_count") / F.col("fraud_rate_denominator")
    )
    .drop("fraud_rate_denominator")
    .orderBy(F.desc("transactions"))
)

display(card_type_analysis)


In [0]:
card_fraud_distribution = card_df.select(
    F.min("fraud_rate").alias("min_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.25)").alias("p25_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.50)").alias("p50_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.75)").alias("p75_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.90)").alias("p90_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.95)").alias("p95_fraud_rate"),
    F.expr("percentile_approx(fraud_rate, 0.99)").alias("p99_fraud_rate"),
    F.max("fraud_rate").alias("max_fraud_rate")
)

display(card_fraud_distribution)


In [0]:
card_df.select(
    "total_transactions",
    "total_spend",
    "fraud_count",
    "fraud_rate",
    "unique_merchants",
    "online_transaction_pct"
).toPandas().corr()


### Card-level findings

Card type

The transaction-weighted fraud rates are:
- Debit: 0.111%
- Credit: 0.131%
- Debit (Prepaid): 0.174%


The prepaid category therefore has the highest observed transaction-weighted fraud rate among the three categories in this dataset.

- There is also an important distinction between the two fraud-rate measures:
avg_card_fraud_rate averages each card's rate equally
- transaction_weighted_fraud_rate calculates total fraudulent transactions / total transactions for the card type


For Power BI, I'd use the transaction-weighted rate when comparing card types because it represents the observed fraud rate across transactions within each category.

Activity

Debit has the highest transaction volume, while Credit has the highest average and median spending per card.
Prepaid cards have substantially lower activity and spending per card, but their observed fraud rate is higher.

Card fraud distribution

The distribution is highly right-skewed:
- Median card fraud rate: 0%
- 75th percentile: 0.159%
- 95th percentile: 0.714%
- 99th percentile: 1.70%
- Maximum: 60%

That 60% maximum is interesting, but a card with relatively few transactions can have a very high rate from a small number of fraudulent transactions. The distribution is more informative than the extreme maximum.


Activity vs fraud

The correlation between card transaction volume and fraud rate is only -0.038, effectively very weak.

So, similar to the user analysis, higher transaction volume does not appear to correspond to higher card-level fraud rate in a simple linear relationship.

Meanwhile:
- transactions ↔ spending: 0.872
- transactions ↔ unique merchants: 0.775
- spending ↔ unique merchants: 0.778

These are strong activity relationships and are more useful for describing card behavior.


Power BI candidates from this section

I'd keep just two main card visuals:

1. Card type comparison
- Cards
- Transactions
- Spend
- Transaction-weighted fraud rate

2. Card activity vs fraud rate
- Scatter plot
- X = transactions
- Y = fraud rate

The second visual is useful precisely because the relationship appears weak; the dashboard can show the actual distribution rather than implying a relationship that isn't there.

### 3. Merchant Behavior

In [0]:
#merchant totals
merchant_totals = merchant_df.select(
    F.count("*").alias("merchants"),
    F.sum("transaction_count").alias("transactions"),
    F.sum("total_amount").alias("total_amount"),
    F.sum("fraud_count").alias("fraud_count")
)

display(merchant_totals)

In [0]:
#merchant concentration

merchant_concentration = (
    merchant_df
    .select(
        "merchant_id",
        "transaction_count",
        "total_amount"
    )
    .orderBy(F.desc("transaction_count"))
    .collect()
)

merchant_count = len(merchant_concentration)

total_merchant_transactions = sum(
    row["transaction_count"]
    for row in merchant_concentration
)

total_merchant_spend = sum(
    row["total_amount"]
    for row in merchant_concentration
)

concentration_levels = [1, 5, 10, 20]

rows = []

for pct in concentration_levels:
    n = max(1, int(merchant_count * pct / 100))

    top_transaction_merchants = merchant_concentration[:n]

    transaction_share = (
        sum(row["transaction_count"] for row in top_transaction_merchants)
        / total_merchant_transactions
    )

    top_spend_merchants = sorted(
        merchant_concentration,
        key=lambda row: row["total_amount"],
        reverse=True
    )[:n]

    spend_share = (
        sum(row["total_amount"] for row in top_spend_merchants)
        / total_merchant_spend
    )

    rows.append(
        (
            pct,
            n,
            transaction_share,
            spend_share
        )
    )

merchant_concentration_df = spark.createDataFrame(
    rows,
    [
        "top_merchant_pct",
        "merchant_count",
        "transaction_share",
        "spend_share"
    ]
)

display(merchant_concentration_df)


### The key observation
- Just 1% of merchants account for approximately 80% of all transactions and 78% of total spending
- The top 5% of merchants account for approximately 91% of transactions and 90% of spending

### Descriptive Conclusion
Transaction activity and spending are both highly concentrated among a relatively small proportion of merchants.

### Power BI implication

A simple concentration chart using the four percentages would communicate this very effectively:

    X-axis: Top merchant percentage

    Y-axis: Share of transactions / spending

    Two series: Transaction share and spend share

### Merchant Online Behavior Distribution

In [0]:
merchant_online_distribution = merchant_df.select(
    F.min("online_transaction_pct").alias("min"),
    F.expr("percentile_approx(online_transaction_pct, 0.25)").alias("p25"),
    F.expr("percentile_approx(online_transaction_pct, 0.50)").alias("median"),
    F.expr("percentile_approx(online_transaction_pct, 0.75)").alias("p75"),
    F.expr("percentile_approx(online_transaction_pct, 0.90)").alias("p90"),
    F.expr("percentile_approx(online_transaction_pct, 0.95)").alias("p95"),
    F.expr("percentile_approx(online_transaction_pct, 0.99)").alias("p99"),
    F.max("online_transaction_pct").alias("max")
)

display(merchant_online_distribution)


In [0]:
merchant_online_buckets = (
    merchant_df
    .select(
        F.when(F.col("online_transaction_pct") == 0, "0%")
         .when(F.col("online_transaction_pct") < 0.25, "1-24%")
         .when(F.col("online_transaction_pct") < 0.50, "25-49%")
         .when(F.col("online_transaction_pct") < 0.75, "50-74%")
         .when(F.col("online_transaction_pct") < 1.00, "75-99%")
         .otherwise("100%")
         .alias("online_bucket")
    )
    .groupBy("online_bucket")
    .agg(
        F.count("*").alias("merchant_count")
    )
)

display(
    merchant_online_buckets.orderBy(
        F.when(F.col("online_bucket") == "0%", 1)
         .when(F.col("online_bucket") == "1-24%", 2)
         .when(F.col("online_bucket") == "25-49%", 3)
         .when(F.col("online_bucket") == "50-74%", 4)
         .when(F.col("online_bucket") == "75-99%", 5)
         .when(F.col("online_bucket") == "100%", 6)
    )
)


What percentage of all transactions represented by the merchant Gold table are online?

In [0]:
merchant_online_summary = merchant_df.select(
    F.sum(
        F.col("transaction_count") *
        F.col("online_transaction_pct") / 100
    ).alias("estimated_online_transactions"),
    F.sum("transaction_count").alias("total_transactions")
).withColumn(
    "transaction_weighted_online_rate",
    F.col("estimated_online_transactions") /
    F.col("total_transactions") * 100
)

display(merchant_online_summary)



Online activity is concentrated among a very small number of merchants, but those merchants account for approximately 11.16% of transaction activity

Power BI
A useful Merchant Behavior visual could show:
- 11.16% — transaction-weighted online rate
- 88.84% — non-online transaction share
- potentially alongside the number of merchants with any online activity: 218 / 100,343 = 0.217%


### Merchant online behavior — interpretation

Online transaction activity is highly concentrated at merchant level. 
Only 218 of 100,343 merchants have a non-zero online transaction share,
while the transaction-weighted online rate is approximately 11.16%.

This indicates that online activity is concentrated among a very small
subset of merchants, although those merchants represent a meaningful
share of overall transaction activity.

The merchant-level percentage distribution and transaction-weighted rate
measure different aspects of online behavior and should not be conflated.


### Merchant fraud-rate distribution

In [0]:
merchant_fraud_distribution = merchant_df.select(
    F.min("fraud_rate").alias("min"),
    F.expr("percentile_approx(fraud_rate, 0.25)").alias("p25"),
    F.expr("percentile_approx(fraud_rate, 0.50)").alias("median"),
    F.expr("percentile_approx(fraud_rate, 0.75)").alias("p75"),
    F.expr("percentile_approx(fraud_rate, 0.90)").alias("p90"),
    F.expr("percentile_approx(fraud_rate, 0.95)").alias("p95"),
    F.expr("percentile_approx(fraud_rate, 0.99)").alias("p99"),
    F.max("fraud_rate").alias("max")
)

display(merchant_fraud_distribution)


Are high merchant fraud rates associated with very low transaction counts?

In [0]:
merchant_fraud_tail = (
    merchant_df
    .select(
        "merchant_id",
        "transaction_count",
        "fraud_count",
        "fraud_rate"
    )
    .orderBy(F.desc("fraud_rate"))
)

display(merchant_fraud_tail.limit(20))


In [0]:
merchant_fraud_volume_summary = (
    merchant_df
    .groupBy(
        F.when(F.col("transaction_count") < 10, "<10")
         .when(F.col("transaction_count") < 50, "10-49")
         .when(F.col("transaction_count") < 100, "50-99")
         .when(F.col("transaction_count") < 500, "100-499")
         .otherwise("500+")
         .alias("transaction_volume_bucket")
    )
    .agg(
        F.count("*").alias("merchant_count"),
        F.sum("transaction_count").alias("transactions"),
        F.sum("fraud_count").alias("fraud_count")
    )
    .withColumn(
        "transaction_weighted_fraud_rate",
        F.col("fraud_count") / F.col("transactions")
    )
)

display(
    merchant_fraud_volume_summary.orderBy(
        F.when(F.col("transaction_volume_bucket") == "<10", 1)
         .when(F.col("transaction_volume_bucket") == "10-49", 2)
         .when(F.col("transaction_volume_bucket") == "50-99", 3)
         .when(F.col("transaction_volume_bucket") == "100-499", 4)
         .otherwise(5)
    )
)


Merchants with lower transaction volumes have higher transaction-weighted observed fraud rates, while high-volume merchants have lower observed fraud rates in this dataset.

## Power BI

merchant transaction volume vs. observed fraud rate visual based on these buckets.

### Merchant fraud-rate interpretation

Merchant-level fraud rates are highly skewed. The median merchant fraud
rate is 0%, while the upper tail contains merchants with very high
observed rates, including 100%.

Inspection of the highest-rate merchants shows that these extreme rates
often occur among merchants with very few transactions. For example,
some merchants have 100% observed fraud rates because all 1, 7, 14, 36,
or 55 observed transactions were fraudulent.

Transaction-volume segmentation provides a more informative view.
Transaction-weighted observed fraud rates decrease across the merchant
volume groups, from 0.843% among merchants with fewer than 10
transactions to 0.111% among merchants with 500 or more transactions.

This is a descriptive association in the observed dataset and should
not be interpreted as a causal relationship.


### Merchant Activity vs. Fraud Rate

In [0]:
merchant_activity_fraud_corr = merchant_df.select(
    F.corr(
        "transaction_count",
        "fraud_rate"
    ).alias("transactions_vs_fraud_rate"),
    F.corr(
        "total_amount",
        "fraud_rate"
    ).alias("spending_vs_fraud_rate"),
    F.corr(
        "transaction_count",
        "online_transaction_pct"
    ).alias("transactions_vs_online_pct")
)

display(merchant_activity_fraud_corr)


### Merchant activity vs. fraud rate

Merchant transaction volume has essentially no linear relationship with
merchant fraud rate in the overall merchant population
(correlation = -0.003). Merchant spending shows a similarly negligible
linear relationship with fraud rate (correlation = -0.002).

Transaction-volume segmentation nevertheless shows higher
transaction-weighted observed fraud rates among low-volume merchants.
This difference is strongly affected by small transaction denominators,
as demonstrated by the presence of merchants with 100% observed fraud
rates despite having only a small number of transactions.

These results indicate that merchant fraud-rate extremes should be
interpreted together with transaction volume rather than in isolation.
The analysis is descriptive and does not establish causality.


##  4. Fraud-Feature Analysis

This section examines descriptive relationships between observed
transaction fraud and the 30-day historical features available in
`gold_fraud_features`.

The analysis compares fraudulent and non-fraudulent transactions across:

- user transaction and fraud history;
- card transaction and fraud history;
- merchant transaction and fraud history.

The objective is to identify descriptive differences that may be useful
for the Power BI presentation layer.

This analysis is not predictive and does not establish causality.

Historical features follow the established 30-day logic:

- historical_timestamp >= current_timestamp - 30 days
- historical_timestamp < current_timestamp

Tied transaction timestamps are treated as simultaneous and are not
artificially ordered. A transaction does not contribute to its own
historical features.


### Establish the observed fraud baseline

In [0]:
fraud_baseline = (
    fraud_df
    .groupBy("is_fraud")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("total_amount")
    )
    .orderBy("is_fraud")
)

display(fraud_baseline)



So observed fraud represents about 0.122% of transactions.

### Compare the nine historical features by fraud status

In [0]:
historical_feature_summary = (
    fraud_df
    .groupBy("is_fraud")
    .agg(
        F.count("*").alias("transaction_count"),

        F.avg("user_30d_transaction_count").alias(
            "avg_user_30d_transaction_count"
        ),
        F.avg("user_30d_fraud_count").alias(
            "avg_user_30d_fraud_count"
        ),
        F.avg("user_30d_fraud_rate").alias(
            "avg_user_30d_fraud_rate"
        ),

        F.avg("card_30d_transaction_count").alias(
            "avg_card_30d_transaction_count"
        ),
        F.avg("card_30d_fraud_count").alias(
            "avg_card_30d_fraud_count"
        ),
        F.avg("card_30d_fraud_rate").alias(
            "avg_card_30d_fraud_rate"
        ),

        F.avg("merchant_30d_transaction_count").alias(
            "avg_merchant_30d_transaction_count"
        ),
        F.avg("merchant_30d_fraud_count").alias(
            "avg_merchant_30d_fraud_count"
        ),
        F.avg("merchant_30d_fraud_rate").alias(
            "avg_merchant_30d_fraud_rate"
        )
    )
    .orderBy("is_fraud")
)

display(historical_feature_summary)


User history

### User history
Fraudulent transactions are associated with users who, on average, have:
- substantially higher historical fraud counts
- substantially higher historical fraud rates
- somewhat lower historical transaction volume


### Card history
The average historical card fraud rate is approximately:
- 0.098% for non-fraud transactions
- 12.599% for fraud transactions

That's a very large descriptive difference.

### Merchant history
Fraudulent transactions are associated with merchants having:
- lower recent transaction volume
- substantially higher recent fraud counts
- substantially higher recent fraud rates


### The most important descriptive observation

There is a very consistent pattern across user, card, and merchant history:

Historical fraud measures show much larger differences between fraudulent and non-fraudulent transactions than historical transaction-volume measures.


### Quantify the differences

In [0]:
historical_comparison = historical_feature_summary.collect()

non_fraud = next(
    row for row in historical_comparison
    if row["is_fraud"] == False
)

fraud = next(
    row for row in historical_comparison
    if row["is_fraud"] == True
)

comparison_rows = [
    (
        "user_30d_transaction_count",
        non_fraud["avg_user_30d_transaction_count"],
        fraud["avg_user_30d_transaction_count"]
    ),
    (
        "user_30d_fraud_count",
        non_fraud["avg_user_30d_fraud_count"],
        fraud["avg_user_30d_fraud_count"]
    ),
    (
        "user_30d_fraud_rate",
        non_fraud["avg_user_30d_fraud_rate"],
        fraud["avg_user_30d_fraud_rate"]
    ),
    (
        "card_30d_transaction_count",
        non_fraud["avg_card_30d_transaction_count"],
        fraud["avg_card_30d_transaction_count"]
    ),
    (
        "card_30d_fraud_count",
        non_fraud["avg_card_30d_fraud_count"],
        fraud["avg_card_30d_fraud_count"]
    ),
    (
        "card_30d_fraud_rate",
        non_fraud["avg_card_30d_fraud_rate"],
        fraud["avg_card_30d_fraud_rate"]
    ),
    (
        "merchant_30d_transaction_count",
        non_fraud["avg_merchant_30d_transaction_count"],
        fraud["avg_merchant_30d_transaction_count"]
    ),
    (
        "merchant_30d_fraud_count",
        non_fraud["avg_merchant_30d_fraud_count"],
        fraud["avg_merchant_30d_fraud_count"]
    ),
    (
        "merchant_30d_fraud_rate",
        non_fraud["avg_merchant_30d_fraud_rate"],
        fraud["avg_merchant_30d_fraud_rate"]
    )
]

historical_comparison_df = spark.createDataFrame(
    comparison_rows,
    [
        "feature",
        "non_fraud_average",
        "fraud_average"
    ]
).withColumn(
    "absolute_difference",
    F.col("fraud_average") - F.col("non_fraud_average")
).withColumn(
    "relative_difference_pct",
    F.when(
        F.col("non_fraud_average") != 0,
        (
            (F.col("fraud_average") - F.col("non_fraud_average"))
            / F.abs(F.col("non_fraud_average"))
        ) * 100
    )
)

display(historical_comparison_df)


Transactions observed as fraudulent are associated with substantially higher 30-day historical fraud counts and fraud rates across users, cards, and merchants.

For Power BI:
Among the three historical fraud-rate measures, the observed averages for fraudulent transactions are:
- User: 5.04%
- Card: 12.60%
- Merchant: 14.18%


The average historical fraud rates are compelling, but averages can be heavily influenced by extreme values.
So let's examine the distribution of historical fraud rates for fraudulent vs. non-fraudulent transactions and let's focus on the three strongest features:
- user_30d_fraud_rate
- card_30d_fraud_rate
- merchant_30d_fraud_rate


In [0]:
historical_fraud_rate_distribution = (
    fraud_df
    .groupBy("is_fraud")
    .agg(
        F.expr(
            "percentile_approx(user_30d_fraud_rate, array(0.25, 0.50, 0.75, 0.90, 0.95, 0.99))"
        ).alias("user_fraud_rate_percentiles"),

        F.expr(
            "percentile_approx(card_30d_fraud_rate, array(0.25, 0.50, 0.75, 0.90, 0.95, 0.99))"
        ).alias("card_fraud_rate_percentiles"),

        F.expr(
            "percentile_approx(merchant_30d_fraud_rate, array(0.25, 0.50, 0.75, 0.90, 0.95, 0.99))"
        ).alias("merchant_fraud_rate_percentiles")
    )
    .orderBy("is_fraud")
)

display(historical_fraud_rate_distribution)

This is a very useful result. It confirms that the differences we saw in the averages are not just being driven by a few extreme observations, particularly for user and card history.

Observed fraudulent transactions are associated with materially higher recent historical fraud rates at the user, card, and merchant levels.